In [ ]:
# ----- 0. Importok -----
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import numpy as np
import time
from datetime import datetime
import re
import os

In [ ]:
# ----- 1. Böngésző indítása -----
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ----- 2. Beolvassuk a linkeket -----
links_df = pd.read_csv("zenga_links.csv")

output_path = "zenga_listings_details.csv"
# Ellenőrizzük, hogy létezik-e már a CSV fájl
if os.path.exists(output_path):
    # Ha létezik, betöltjük a meglévő adatokat
    existing_df = pd.read_csv(output_path)

all_data = []

In [ ]:
# ----- 3. Hirdetések feldolgozása (fejlettebb) -----
def extract_number(text):
    """Kiveszi a számot szövegből, pl. '90 millió Ft' -> 90, '113 m²' -> 113"""
    if text is None:
        return None
    text = text.replace("\xa0", " ").replace(".", "")
    match = re.search(r'\d+', text)
    if match:
        return int(match.group())
    return None

for idx, row in links_df.iterrows():
    url = row["url"]
    if (url in existing_df["url"].unique()) or pd.isna(url):
        continue
    else:
        driver.get(url)
        print(f"{idx}/{len(links_df)}")
        print(f"\nNyitva: {url}")
        
        time.sleep(3)  # várakozás a betöltődésre
        
        # ---- Title ----
        try:
            title = driver.find_element(By.CSS_SELECTOR, 'h1[data-id="h1"]').text.strip()
            print(f"Title: Találva -> {title}")
        except:
            title = None
            print("Title: Nem található")
        
        # ---- Price ----
        try:
            price_raw = driver.find_element(By.CSS_SELECTOR, '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-main-details > div > div.d-flex.flex-column.flex-gap-sm-16.flex-gap-8 > div.row.ng-star-inserted > div.col-8 > oom-portal-advert-price > div > div > div.fc-black-2.fs-32.fw-900.text-nowrap').text.strip()
            price = extract_number(price_raw)
            print(f"Price: Találva -> {price}")
        except:
            price = None
            print("Price: Nem található")
        
        # ---- Area (m²) ----
        try:
            area_raw = driver.find_element(By.CSS_SELECTOR, 'div[data-cy="advert-details-first-param"]').text.strip()
            area_m2 = extract_number(area_raw)
            print(f"Area: Találva -> {area_m2}")
        except:
            area_m2 = None
            print("Area: Nem található")
        
        # ---- Floor ----
        try:
            floor = driver.find_element(By.CSS_SELECTOR, '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-main-details > div > div.d-flex.flex-row.justify-content-between.align-items-center.ng-star-inserted > div > div:nth-child(3) > div.text-nowrap.fc-black-2.fs-20.fw-bold.text-center.ng-star-inserted').text.strip()
            print(f"Floor: Találva -> {floor}")
        except:
            floor = None
            print("Floor: Nem található")
        
        # ---- Rooms ----
        try:
            rooms_raw = driver.find_element(By.CSS_SELECTOR, 'div[data-cy="advert-details-second-param"]').text.strip()
            rooms = extract_number(rooms_raw)
            print(f"Rooms: Találva -> {rooms}")
        except:
            rooms = None
            print("Rooms: Nem található")
        
        # ---- Location ----
        try:
            location = driver.find_element(By.CSS_SELECTOR, 'button[data-cy="advert-map-map-btn"] span.fs-16').text.strip()
            print(f"Location: Találva -> {location}")
        except:
            location = None
            print("Location: Nem található")
        
        # ---- Dinamikus tulajdonságok ----
        properties = {}
        try:
            prop_items = driver.find_elements(By.CSS_SELECTOR, 'div[data-cy="advert-details-param-list-item"]')
            for item in prop_items:
                try:
                    key_elem = item.find_element(By.CSS_SELECTOR, '.col-12.d-flex span')
                    val_elem = item.find_element(By.CSS_SELECTOR, '.col-12.fw-bold')
                    key = key_elem.text.strip().replace(":", "")
                    val = val_elem.text.strip()
                    properties[key] = val
                except:
                    continue
            print(f"Tulajdonságok: {len(properties)} db találat")
        except:
            print("Tulajdonságok: Nem található")

        # ---- Lekérdezés dátuma ----
        scrape_date = datetime.today().strftime('%Y-%m-%d')

        # ---- Adat gyűjtése ----
        all_data.append({
            "url": url,
            "title": title,
            "price": price,
            "area_m2": area_m2,
            "floor": floor,
            "rooms": rooms,
            "location": location,
            **properties,  # ide kerül minden dinamikus property
            "scrape_date": scrape_date
        })

        time.sleep(2)

In [ ]:
# ----- 4. Eredmények tisztítása és mentése -----
df = pd.DataFrame(all_data)

# Másolat a tisztításhoz
df_clean = df.copy()

# 1. Felesleges oszlopok törlése
df_clean = df_clean.loc[:, ~df_clean.columns.str.contains('^Unnamed')]

# 2. Mértékegységek eltávolítása és számokká alakítás
def extract_number(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace("\xa0", " ").replace(",", ".")
    # magyar millió és ezer jelölés kezelése
    if "millió" in val.lower():
        try:
            num = float(val.lower().split("millió")[0].strip())
            return num * 1_000_000
        except:
            return np.nan
    # csak számjegyek
    num_str = ''.join(ch for ch in val if ch.isdigit() or ch == '.')
    try:
        return float(num_str)
    except:
        return np.nan

numeric_cols = ["price", "area_m2", "rooms", "floors_total", "year_built", "balcony"]
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(extract_number)

# 3. Location hibás sorok javítása (ha location üres vagy furcsa, title-ből kinyerés)
def fix_location(row):
    loc = row["location"]
    if pd.isna(loc) or "leírás" in str(loc).lower():
        title = str(row["title"])
        if "," in title:
            parts = title.split(",")
            return parts[0].replace("Eladó", "").strip()
        return np.nan
    return loc.strip()

if "location" in df_clean.columns and "title" in df_clean.columns:
    df_clean["location"] = df_clean.apply(fix_location, axis=1)

# 4. Hiányzó értékek egységes kezelése
df_clean = df_clean.replace(["", " ", "nan", "None", "Nem adta meg a hirdető"], np.nan)

# 5. Szöveges oszlopok egységesítése (kisbetűsítés, whitespace eltávolítás)
text_cols = df_clean.select_dtypes(include=["object"]).columns
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Mentés
# Ellenőrizzük, hogy létezik-e már a CSV fájl
if os.path.exists(output_path):
    # Az új adatokat hozzáfűzzük
    updated_df = pd.concat([existing_df, pd.DataFrame(all_data)], ignore_index=True)
    # Eltávolítjuk a duplikátumokat URL alapján
    updated_df = updated_df.drop_duplicates(subset=["url"], keep="last")
else:
    # Ha nem létezik, létrehozzuk az új DataFrame-t
    updated_df = pd.DataFrame(all_data)

# Mentés CSV-be
updated_df.to_csv(output_path, index=False)
print(f"Adatok frissítve. Összesen {len(updated_df)} hirdetés van a CSV-ben.")

In [ ]:
# ----- 5. Böngésző bezárása -----
driver.quit()